In [1]:
import numpy as np
from PIL import Image, ImageFilter


def angular_gaussian(theta, center, width):
    """
    Wrapped Gaussian around an angle.
    This makes smooth circular arc segments.
    """
    d = np.angle(np.exp(1j * (theta - center)))
    return np.exp(-0.5 * (d / width) ** 2)


def alpha_composite(dst, src_rgb, src_a):
    """
    Alpha-composite one colored layer over an RGBA image.

    dst: float RGBA array, shape (H, W, 4), values in [0, 1]
    src_rgb: RGB tuple, values in [0, 1]
    src_a: alpha array, shape (H, W), values in [0, 1]
    """
    src_a = np.clip(src_a, 0, 1)[..., None].astype(np.float32)
    src_rgb = np.asarray(src_rgb, dtype=np.float32).reshape(1, 1, 3)

    dst_rgb = dst[..., :3]
    dst_a = dst[..., 3:4]

    out_a = src_a + dst_a * (1 - src_a)
    out_rgb = (
        src_rgb * src_a + dst_rgb * dst_a * (1 - src_a)
    ) / np.maximum(out_a, 1e-6)

    dst[..., :3] = out_rgb
    dst[..., 3:4] = out_a

    return dst


def generate_deepprior_ring(
    size=1800,
    radius=0.53,
    thickness=0.045,
    seed=11,
    output="deepprior_gold_orange_ring_transparent.png",
    preview_output="deepprior_gold_orange_ring_preview_white.png",
):
    """
    Generate an abstract glowing circular ring with transparent background.

    Output:
    - transparent PNG for logo/website use
    - white-background preview for inspection
    """

    rng = np.random.default_rng(seed)

    # Coordinate system
    y, x = np.mgrid[-1:1:complex(size), -1:1:complex(size)]

    # Slight ellipse distortion so the ring feels more organic
    xe = x * 1.00
    ye = y * 1.03

    r = np.sqrt(xe**2 + ye**2)
    theta = np.arctan2(ye, xe)

    # Empty transparent canvas
    rgba = np.zeros((size, size, 4), dtype=np.float32)

    # Premium gold/orange/black palette
    gold = (1.00, 0.70, 0.22)
    orange = (1.00, 0.42, 0.07)
    pale_gold = (1.00, 0.88, 0.48)
    white_hot = (1.00, 0.97, 0.82)
    bronze = (0.62, 0.38, 0.16)
    charcoal = (0.05, 0.045, 0.04)
    smoky_gray = (0.24, 0.22, 0.20)

    # ------------------------------------------------------------
    # 1. Soft outer aura
    # ------------------------------------------------------------
    base_ring = np.exp(-0.5 * ((r - radius) / 0.115) ** 2)

    uneven = (
        0.55
        + 0.25 * np.sin(2.8 * theta + 1.0)
        + 0.16 * np.sin(6.1 * theta - 0.7)
    )
    uneven = np.clip(uneven, 0.0, 1.0)

    rgba = alpha_composite(
        rgba,
        pale_gold,
        0.08 * base_ring * uneven,
    )

    # ------------------------------------------------------------
    # 2. Main broad glowing arcs
    # ------------------------------------------------------------
    arc_defs = [
        # angle, width, opacity, color
        (np.deg2rad(35), 0.60, 0.40, gold),
        (np.deg2rad(58), 0.36, 0.28, orange),
        (np.deg2rad(230), 0.58, 0.46, gold),
        (np.deg2rad(260), 0.40, 0.35, orange),
        (np.deg2rad(170), 0.34, 0.30, smoky_gray),
        (np.deg2rad(300), 0.33, 0.26, charcoal),
        (np.deg2rad(85), 0.25, 0.22, bronze),
    ]

    for center, width, opacity, color in arc_defs:
        radial = np.exp(-0.5 * ((r - radius) / thickness) ** 2)
        arc = angular_gaussian(theta, center, width)

        # Small wave pattern gives the "flashing / shimmering" feeling
        streak = 0.78 + 0.22 * np.sin(22 * theta + 7 * r)

        alpha = opacity * radial * arc * streak
        rgba = alpha_composite(rgba, color, alpha)

    # ------------------------------------------------------------
    # 3. Thin bright flashing strokes
    # ------------------------------------------------------------
    for _ in range(22):
        center = rng.uniform(-np.pi, np.pi)
        width = rng.uniform(0.018, 0.060)

        local_radius = radius + rng.normal(0, 0.025)
        local_thick = rng.uniform(0.004, 0.010)
        opacity = rng.uniform(0.18, 0.55)

        radial = np.exp(-0.5 * ((r - local_radius) / local_thick) ** 2)
        arc = angular_gaussian(theta, center, width)

        color = white_hot if rng.random() < 0.45 else pale_gold
        rgba = alpha_composite(rgba, color, opacity * radial * arc)

    # ------------------------------------------------------------
    # 4. Smoky black broken arcs
    # ------------------------------------------------------------
    for _ in range(12):
        center = rng.choice(
            [
                np.deg2rad(160),
                np.deg2rad(285),
                np.deg2rad(320),
            ]
        ) + rng.normal(0, 0.18)

        width = rng.uniform(0.05, 0.18)
        local_radius = radius + rng.normal(0, 0.035)
        local_thick = rng.uniform(0.018, 0.042)
        opacity = rng.uniform(0.08, 0.25)

        radial = np.exp(-0.5 * ((r - local_radius) / local_thick) ** 2)
        arc = angular_gaussian(theta, center, width)

        rgba = alpha_composite(rgba, charcoal, opacity * radial * arc)

    # ------------------------------------------------------------
    # 5. Clear center, so it remains a ring rather than a blob
    # ------------------------------------------------------------
    inner_clear = 1 - np.exp(-0.5 * ((r - radius * 0.70) / 0.18) ** 2)
    rgba[..., 3] *= np.clip(inner_clear + 0.15, 0, 1)

    # ------------------------------------------------------------
    # 6. Fade outer edges for a soft logo aura
    # ------------------------------------------------------------
    fade = np.exp(-0.5 * (r / 0.85) ** 8)
    rgba[..., 3] *= fade

    # Convert to image
    img = Image.fromarray(
        (np.clip(rgba, 0, 1) * 255).astype(np.uint8),
        mode="RGBA",
    )

    # Final blur: increase this for a softer website-background aura
    img = img.filter(ImageFilter.GaussianBlur(radius=0.7))

    # Save transparent logo
    img.save(output)

    # Save white-background preview
    white = Image.new("RGBA", img.size, (255, 255, 255, 255))
    white.alpha_composite(img)
    white.convert("RGB").save(preview_output, quality=95)

    print(f"Saved transparent logo: {output}")
    print(f"Saved white preview:     {preview_output}")


if __name__ == "__main__":
    generate_deepprior_ring(
        size=1800,
        radius=0.53,
        thickness=0.045,
        seed=11,
        output="deepprior_gold_orange_ring_transparent.png",
        preview_output="deepprior_gold_orange_ring_preview_white.png",
    )

Saved transparent logo: deepprior_gold_orange_ring_transparent.png
Saved white preview:     deepprior_gold_orange_ring_preview_white.png
